# 02 — Results and Tables

This notebook loads simulation outputs and constructs manuscript Tables 1–3
plus the derived Table 2 (unsafe systems per 100 deployments).

In [ ]:
import pandas as pd
import json
from pathlib import Path
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

## Table 1: Threshold Profile Performance

In [ ]:
table1 = pd.read_csv('outputs/tables/table1_thresholds.csv')
print('Table 1: Governance threshold profile performance')
print('(20% unsafe base rate, moderate audit conditions, 5 replicates)')
print(table1.to_string(index=False))

## Non-Compensatory vs Compensatory Comparison

In [ ]:
comp = pd.read_csv('outputs/raw/compensatory_comparison.csv')
nc_det = comp['nc_detection'].mean()
c_det = comp['c_detection'].mean()
nc_fn = comp['nc_fn_harm'].mean()
c_fn = comp['c_fn_harm'].mean()
nc_thr = comp['nc_throughput'].mean()
c_thr = comp['c_throughput'].mean()
fold = c_fn / nc_fn

print(f'NC detection: {nc_det:.4f} ({nc_det*100:.1f}%)')
print(f'Comp detection: {c_det:.4f} ({c_det*100:.1f}%)')
print(f'NC FN harm: {nc_fn:.2f}')
print(f'Comp FN harm: {c_fn:.2f}')
print(f'Fold reduction: {fold:.1f}')
print(f'NC throughput: {nc_thr:.4f} ({nc_thr*100:.1f}%)')
print(f'Comp throughput: {c_thr:.4f} ({c_thr*100:.1f}%)')

## Table 2: Unsafe Systems per 100 Deployments (Derived)

In [ ]:
# Derive Table 2: (1 - detection) * base_rate * 100
regimes = {
    'NC Moderate': {'det': round(nc_det * 1000) / 1000, 'thr': float(table1[table1['profile']=='moderate']['throughput']), 'fn': float(table1[table1['profile']=='moderate']['fn_harm'])},
    'NC Permissive': {'det': float(table1[table1['profile']=='permissive']['detection_rate']), 'thr': float(table1[table1['profile']=='permissive']['throughput']), 'fn': float(table1[table1['profile']=='permissive']['fn_harm'])},
    'Compensatory': {'det': round(c_det * 1000) / 1000, 'thr': round(c_thr * 1000) / 1000, 'fn': round(c_fn, 1)},
}
rows = []
for name, v in regimes.items():
    row = {'Regime': name, 'Detection': f"{v['det']*100:.1f}%"}
    for br in [0.10, 0.20, 0.30]:
        row[f'{int(br*100)}% base'] = round((1 - v['det']) * br * 100, 2)
    row['Throughput'] = f"{v['thr']*100:.1f}%" if isinstance(v['thr'], float) else v['thr']
    row['FN Harm'] = v['fn']
    rows.append(row)

table2 = pd.DataFrame(rows)
print('Table 2: Expected unsafe AI systems per 100 deployment decisions')
print(table2.to_string(index=False))
table2.to_csv('outputs/tables/table2_per100_deployments.csv', index=False)
print('\nSaved to outputs/tables/table2_per100_deployments.csv')

## Table 3: Pareto Frontier Summary

In [ ]:
table3 = pd.read_csv('outputs/tables/table2_pareto.csv')
pareto = pd.read_csv('outputs/processed/pareto_solutions.csv')
sweet = pareto[pareto['in_sweet_spot'] == True]

print(f'Table 3: Pareto frontier summary ({len(pareto)} non-dominated solutions)')
print(table3.to_string(index=False))
print(f'\nSweet-spot solutions: {len(sweet)} ({100*len(sweet)/len(pareto):.0f}%)')

## Manuscript Numerical Assertions

In [ ]:
# Verify critical manuscript values
assert abs(nc_det - 0.998) < 0.002, f'NC detection {nc_det} not ~0.998'
assert abs(c_det - 0.772) < 0.002, f'Comp detection {c_det} not ~0.772'
assert abs(nc_fn - 2.1) < 0.5, f'NC FN harm {nc_fn} not ~2.1'
assert abs(c_fn - 323.1) < 5, f'Comp FN harm {c_fn} not ~323.1'
assert len(pareto) == 60, f'Pareto count {len(pareto)} not 60'
assert len(sweet) == 20, f'Sweet-spot count {len(sweet)} not 20'

summary = json.loads(Path('outputs/processed/simulation_summary.json').read_text())
life = summary['lifecycle_summary']
assert abs(life['mean_final_decay'] - 0.033) < 0.002
assert abs(life['mean_total_harm'] - 5.05) < 0.1
assert abs(life['re_audit_rate'] - 0.101) < 0.005

print('All manuscript numerical assertions PASSED')